# Try the LangChain agent

Start the local server with `scripts/start_llm.ps1` (Windows) or `bash scripts/start_llm.sh` (Linux) in a separate terminal. Select a Python kernel with `requirements.txt` installed, then run these cells in order.

This experiment uses **fictional listings only**. It prints tool calls, results and an unvalidated answer. It has no database access, masking, policy guards or grounding checks.


## 1. Imports


In [ ]:
import json

from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langsmith import tracing_context

from langchain_agent.workflow import local_model




## 2. Edit the question and fictional data

The default search should match listing 1: four guests in Neubau at EUR 120. Use fictional input only.


In [ ]:
# Edit this question to try another fictional listing search.
QUESTION = "Find fictional listings in Neubau for at least 4 guests under 150 EUR."

DEMO_LISTINGS = [
    {"id": 1, "district": "Neubau", "guests": 4, "price_eur": 120},
    {"id": 2, "district": "Neubau", "guests": 2, "price_eur": 80},
    {"id": 3, "district": "Leopoldstadt", "guests": 6, "price_eur": 140},
]


@tool
def search_demo_listings(district: str, guests: int, max_price: float) -> list[dict]:
    """Find fictional listings by district, minimum guest capacity and maximum EUR price."""
    return [row for row in DEMO_LISTINGS
            if row["district"].casefold() == district.casefold()
            and row["guests"] >= guests and row["price_eur"] <= max_price]




## 3. Define the bounded agent loop


In [ ]:
def main():
    # This alias and URL match the repository's local server launchers.
    model = local_model(model="vienna-stay")
    agent = model.bind_tools([search_demo_listings])
    messages = [
        SystemMessage(
            "You are testing fictional Vienna listings. Use search_demo_listings "
            "to answer listing questions. All data is invented. After receiving "
            "the tool result, give a short answer using only that result."
        ),
        HumanMessage(QUESTION),
    ]
    print("Fictional-data demo (no database, privacy guards or grounding checks).")
    print("Question:", QUESTION)
    calls = 0
    # Disable environment tracing and keep this local experiment bounded.
    with tracing_context(enabled=False):
        for _ in range(4):
            response = agent.invoke(messages, config={"callbacks": []})
            if response.invalid_tool_calls:
                raise ValueError("The model returned a malformed tool call.")
            if not response.tool_calls:
                print("Answer (unvalidated):", response.content)
                if calls == 0:
                    print("No tool was called; this run did not verify tool calling.")
                return
            if calls + len(response.tool_calls) > 4:
                print("Stopped at the tool-call limit.")
                return
            messages.append(response)
            for call in response.tool_calls:
                if call["name"] != search_demo_listings.name:
                    raise ValueError("The model requested an unknown tool.")
                result = search_demo_listings.invoke(call["args"], config={"callbacks": []})
                print("Tool:", call["name"], call["args"])
                print("Result:", json.dumps(result, ensure_ascii=False))
                messages.append(ToolMessage(
                    content=json.dumps(result), tool_call_id=call["id"],
                ))
                calls += 1
    print("Stopped at the model-round limit.")




## 4. Run inference

The server must be ready at `http://127.0.0.1:8080/v1` with model alias `vienna-stay`. Rerun this cell for a fresh request after editing and rerunning cell 2.


In [ ]:
main()
